In [ ]:
import numpy as np
import os
import scipy
from load_data_function import load_data,save_data
import re
from load_data_function import fig_plot,battery_soh_plot,smooth_soh
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter1d

In [ ]:
HNEL_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data\HNEL_dataset'
package_list=['package_1']
HNEL_data={}
HNEL_SOH={}
for i,package in enumerate(package_list):

    print(f'package: {package}')
    battery_list=os.listdir(HNEL_path)

    print(battery_list)

    package1={}
    package2={}
    for j,battery in enumerate(battery_list):
        print(f'battery: {battery}')
        battery_path=os.path.join(HNEL_path,battery)
        battery_data = pd.read_csv(battery_path).dropna(subset=['Discharge_Capacity (Ah)', 'Charge_Capacity (Ah)'])

        print(battery_data.shape)

        voltage=[]
        current=[]
        time=[]
        capacity=[]
        package1[f'battery_{j+1}']=[]
        package2[f'battery_{j+1}']=[]
        cycle_num= battery_data['Cycle_Index'].unique()
        cycle_num=sorted(cycle_num)
        #print(cycle_num)
        for k in cycle_num:
            cycle_data=battery_data[battery_data['Cycle_Index']==k]
            voltage=cycle_data['Voltage (V)'].values.reshape(1,-1)

            current=cycle_data['Current (A)'].values.reshape(1,-1)
            time_segment = cycle_data['Test_Time (s)'].values.reshape(1,-1)
            time=time_segment  # 转换为秒
            #time=time.reshape(1,-1)
            discharge_capacity=cycle_data['Discharge_Capacity (Ah)'].values.reshape(1,-1)
            charge_capacity=cycle_data['Charge_Capacity (Ah)'].values.reshape(1,-1)
            discharge_capacity=np.float32(discharge_capacity)
            #charge_capacity=np.float32(charge_capacity)
            #capacity=np.concatenate((discharge_capacity,charge_capacity),axis=1)
            if discharge_capacity.shape[1]<100:
                continue
            print(discharge_capacity.shape)
            capacity_max=np.max(discharge_capacity)
            soh=capacity_max/2.8
            print(soh)
            package1[f'battery_{j+1}'].append(np.concatenate((voltage,current,time),axis=0))
            package2[f'battery_{j+1}'].append(soh)
    HNEL_data[f'package_{i+1}']=package1
    HNEL_SOH[f'package_{i+1}']=package2

In [ ]:
battery_soh_plot(HNEL_SOH,HNEL_SOH['package_1'].keys(),package='package_1')

In [ ]:
smooth_HNEL_SOH=smooth_soh(HNEL_SOH,method='moving_average', sigma=4)
package='package_1'
battery_soh_plot(smooth_HNEL_SOH,smooth_HNEL_SOH[package].keys(),package)

In [ ]:
fig_plot(HNEL_SOH['package_1']['battery_14'])

In [ ]:
print(HNEL_SOH['package_1']['battery_12'])

In [ ]:
save_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\HNEL_dataset'
#save_data(HNEL_data,os.path.join(save_path,'HNEL_data.pkl'))
#save_data(smooth_HNEL_SOH,os.path.join(save_path,'HNEL_SOH.pkl'))
HNEL_data=load_data(os.path.join(save_path,'HNEL_data.pkl'))
HNEL_SOH=load_data(os.path.join(save_path,'HNEL_SOH.pkl'))


In [ ]:
fig_plot(HNEL_data['package_1']['battery_1'][0][1])

In [ ]:
for package in HNEL_data.keys():
    for battery in HNEL_data[package].keys():
        if len(HNEL_data[package][battery])!= len(HNEL_SOH[package][battery]):
            print(f'battery {battery} has different length of data and SOH,data length: {len(HNEL_data[package][battery])}, SOH length: {len(HNEL_SOH[package][battery])}')


In [ ]:
for package in HNEL_data.keys():
    for battery in HNEL_data[package].keys():
        HNEL_data[package][battery] = HNEL_data[package][battery][3:][:][:]

#save_data(HNEL_data,os.path.join(save_path,'HNEL_data.pkl'))


In [ ]:
for package in HNEL_data.keys():
    for battery in HNEL_data[package].keys():
        if len(HNEL_data[package][battery])!= len(HNEL_SOH[package][battery]):
            print(f'battery {battery} has different length of data and SOH,data length: {len(HNEL_data[package][battery])}, SOH length: {len(HNEL_SOH[package][battery])}')